In [1]:
import csv
import glob
import json
import os

In [2]:
def load_people_mapping(people_csv_path):
    """
    Loads people.csv which contains columns 'identifier' and 'key_cricinfo'.
    Uses the 'identifier' (the hex value) as the key and maps it to the corresponding 'key_cricinfo'.
    If key_cricinfo is a float ending with .0, it is converted to an integer string.

    Returns a dictionary: {identifier: key_cricinfo, ...}
    """
    mapping = {}
    with open(people_csv_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            identifier = row.get('identifier', '').strip()
            key_val = row.get('key_cricinfo')
            if key_val is not None:
                try:
                    num = float(key_val)
                    key_val = str(int(num)) if num.is_integer() else str(num)
                except ValueError:
                    pass
            mapping[identifier] = key_val
    return mapping


def parse_info_file(info_file_path, people_mapping):
    """
    Processes a single match info CSV file named like {match_id}_info.csv.

    Steps:
      1. Extracts the match_id from the filename (if it's a float string ending with .0, convert it to int).
      2. Reads team names from the first two "info,team,TEAMNAME" rows.
      3. Collects umpire names from "info,umpire,UMPIRE_NAME", "info,reserve_umpire,UMPIRE_NAME",
         "info,tv_umpire,UMPIRE_NAME", and "info,match_referee,UMPIRE_NAME" rows.
      4. Builds a mapping of player names to teams from "info,player,TEAM,PLAYER_NAME" and "info,players,TEAM,PLAYER_NAME" rows.
      5. Processes "info,registry,people,PLAYER_NAME,HEX_IDENTIFIER" rows to get (player_name, hex_identifier) tuples.
      6. For each registry entry (skipping umpires), determines the team from step 4 and uses the hex_identifier
         to look up the corresponding key_cricinfo from people_mapping.
      7. Groups the key_cricinfo values into team1 and team2 playing XI lists.

    Returns a dictionary with keys:
      - match_id
      - team1_name
      - team2_name
      - team1_players: list of key_cricinfo values for team1
      - team2_players: list of key_cricinfo values for team2

    If fewer than two teams are found, returns None.
    """
    base = os.path.basename(info_file_path)
    match_id_str = base.split('_info.csv')[0]
    try:
        num = float(match_id_str)
        match_id = int(num) if num.is_integer() else match_id_str
    except ValueError:
        match_id = match_id_str

    teams = []  # Collect team names from "info,team" rows
    umpires = set()  # Collect umpire names from various umpire-related rows
    player_team_map = {}  # Maps player name to team (from "info,player" or "info,players" rows)
    registry_players = []  # List of tuples (player_name, hex_identifier) from "info,registry,people" rows

    with open(info_file_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            if not row:
                continue
            if row[0] == 'info':
                if row[1] == 'team' and len(row) >= 3:
                    teams.append(row[2])
                elif row[1] in ['umpire', 'reserve_umpire', 'tv_umpire', 'match_referee'] and len(row) >= 3:
                    umpires.add(row[2])
                elif row[1] in ['player', 'players'] and len(row) >= 4:
                    # row format: info,player(s),TEAM,PLAYER_NAME
                    team_name = row[2]
                    player_name = row[3]
                    if player_name not in player_team_map:
                        player_team_map[player_name] = team_name
                elif row[1] == 'registry' and len(row) >= 5 and row[2] == 'people':
                    # row format: info,registry,people,PLAYER_NAME,HEX_IDENTIFIER
                    registry_players.append((row[3], row[4].strip()))

    if len(teams) < 2:
        return None

    team1_name, team2_name = teams[0], teams[1]
    team1_players = []
    team2_players = []

    for player_name, player_hex in registry_players:
        # Skip players listed as umpires.
        if player_name in umpires:
            continue

        # Determine team assignment using the player mapping.
        if player_name not in player_team_map:
            continue
        team_assigned = player_team_map[player_name]

        # Look up the key_cricinfo using the hex identifier.
        if player_hex not in people_mapping:
            continue
        key_cric = people_mapping[player_hex]

        if team_assigned == team1_name:
            team1_players.append(key_cric)
        elif team_assigned == team2_name:
            team2_players.append(key_cric)

    return {
        "match_id": match_id,
        "team1_name": team1_name,
        "team2_name": team2_name,
        "team1_players": team1_players,
        "team2_players": team2_players
    }


def process_matches(people_csv, info_dir, output_csv):
    """
    Processes all match info CSV files in the specified directory.

    Writes an output CSV with five columns:
      match_id, team1_name, team1_playing_xi, team2_name, team2_playing_xi.

    The playing XI for each team is stored as a JSON array of key_cricinfo values.
    """
    people_mapping = load_people_mapping(people_csv)

    with open(output_csv, mode='w', newline='', encoding='utf-8') as outcsv:
        fieldnames = ['match_id', 'team1_name', 'team1_playing_xi', 'team2_name', 'team2_playing_xi']
        writer = csv.DictWriter(outcsv, fieldnames=fieldnames)
        writer.writeheader()

        info_files = glob.glob(os.path.join(info_dir, '*_info.csv'))
        for info_file in info_files:
            result = parse_info_file(info_file, people_mapping)
            if result is None:
                continue
            writer.writerow({
                "match_id": result["match_id"],
                "team1_name": result["team1_name"],
                "team1_playing_xi": json.dumps(result["team1_players"]),
                "team2_name": result["team2_name"],
                "team2_playing_xi": json.dumps(result["team2_players"])
            })

In [3]:
# Example usage (uncomment and adjust the paths as needed):
people_csv_path = r"../../../data/register_data/people.csv"
info_directory = r"D:\ED\TrashPL\data\ssm_male_csv2"
output_csv_path = r"../../../data/register_data/playing_xi/ssm_playing_xi.csv"
process_matches(people_csv_path, info_directory, output_csv_path)


In [4]:
import pandas as pd
df = pd.read_csv(output_csv_path)

In [5]:
df["team1_name"].unique(), df["team2_name"].unique()


(array(['Auckland', 'Canterbury', 'Northern Districts',
        'Central Districts', 'Wellington', 'Otago'], dtype=object),
 array(['Central Districts', 'Otago', 'Auckland', 'Canterbury',
        'Wellington', 'Northern Districts'], dtype=object))

In [6]:
df.to_csv(output_csv_path, index=False)

In [7]:
df

,match_id,team1_name,team1_playing_xi,team2_name,team2_playing_xi
0,1118534,Auckland,"[""1129328"", ""1125536"", ""559061"", ""232359"", ""35...",Central Districts,"[""450860"", ""854909"", ""440523"", ""440517"", ""3625..."
1,1118535,Canterbury,"[""36993"", ""311158"", ""547764"", ""526021"", ""37989...",Otago,"[""384355"", ""38927"", ""36947"", ""331375"", ""547766..."
2,1118537,Northern Districts,"[""36948"", ""315623"", ""36194"", ""547756"", ""4493"",...",Auckland,"[""1129328"", ""1125536"", ""559061"", ""232359"", ""35..."
3,1118538,Central Districts,"[""595783"", ""440520"", ""854909"", ""440523"", ""5083...",Canterbury,"[""36993"", ""1030549"", ""311158"", ""1124470"", ""547..."
4,1118540,Northern Districts,"[""36948"", ""36194"", ""547756"", ""4493"", ""381743"",...",Wellington,"[""326973"", ""379140"", ""37696"", ""226493"", ""38141..."
...,...,...,...,...,...
236,1452620,Canterbury,"[""526021"", ""379890"", ""381743"", ""962831"", ""5590...",Central Districts,"[""1353679"", ""707193"", ""854909"", ""1343344"", ""44..."
237,580784,Wellington,"[""4427"", ""331368"", ""38373"", ""7502"", ""38920"", ""...",Otago,"[""38264"", ""36947"", ""331375"", ""36331"", ""547766""..."
238,665677,Auckland,"[""384355"", ""55395"", ""18048"", ""355266"", ""547768...",Otago,"[""38264"", ""36947"", ""331375"", ""547766"", ""37762""..."
239,665701,Northern Districts,"[""36948"", ""38924"", ""38927"", ""553800"", ""381743""...",Canterbury,"[""36993"", ""36950"", ""4493"", ""277911"", ""226493"",..."


In [8]:
empty_team1 = df[df["team1_playing_xi"] == "[]"]
print(len(empty_team1))

0
